# CEMANEIGE + GR4J Example

This notebook demonstrates how to use CEMANEIGE as a preprocessing wrapper for the GR4J model.

- CEMANEIGE: Transforms raw precipitation/temperature into snow-adjusted liquid precipitation
- GR4J: Uses the snow-adjusted precipitation to simulate runoff


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from hydrogr import CemaNeige, ModelGr4j
from plotly.subplots import make_subplots

## Prepare input data

In [2]:
data_path = Path.cwd().parent / "data"
df = pd.read_csv(data_path / "L0123002.csv", parse_dates=True, header=1)
df.columns = [
    "date",
    "precipitation",
    "temperature",
    "evapotranspiration",
    "flow",
    "flow_mm",
]
df.set_index("date", inplace=True)
df.index = pd.to_datetime(df.index, format="%d/%m/%Y")

## CEMANEIGE without hysteresis

In [3]:
cemaneige_params = {
    "X1": 0.962,  # Snowpack thermal coefficient [0-1]
    "X2": 2.249,  # Melt factor [mm/(°C·day)]
}
cemaneige = CemaNeige(
    parameters=cemaneige_params,
    hysteresis=False,
)

In [4]:
# Create preprocessed input for GR4J :
snowmelt_results = cemaneige.run(df)
gr_inputs = df.copy()
gr_inputs["precipitation"] = snowmelt_results["precipitation"]

In [5]:
fig = make_subplots(rows=3, cols=1, row_heights=[0.2, 0.6, 0.2], shared_xaxes=True)

fig.add_trace(
    go.Scatter(x=df.index, y=df["temperature"], name="Temperature"), row=1, col=1
)
fig.update_yaxes(title="°C", row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df["precipitation"], name="Raw"), row=2, col=1)
fig.add_trace(
    go.Scatter(x=gr_inputs.index, y=gr_inputs["precipitation"], name="Snow-Adjusted"),
    row=2,
    col=1,
)
fig.update_yaxes(title="mm")

snow_diff = df["precipitation"] - gr_inputs["precipitation"]
fig.add_trace(
    go.Scatter(x=df.index, y=snow_diff, name="Precipitation retained as snow"),
    row=3,
    col=1,
)
fig.update_yaxes(title="mm")

fig.update_layout(title="Raw vs Snow-Adjusted Precipitation")
fig.show()

In [6]:
# Initialize GR4J with default parameters
gr4j_params = {
    "X1": 408.774,  # Production store capacity [mm]
    "X2": 2.646,  # Inter-catchment exchange [mm/d]
    "X3": 131.264,  # Routing store capacity [mm]
    "X4": 1.174,  # Unit hydrograph time constant [d]
}
gr4j = ModelGr4j(parameters=gr4j_params)
gr_outputs = gr4j.run(inputs=gr_inputs)

gr4j_no_snow = ModelGr4j(parameters=gr4j_params)
gr_outputs_no_snow = gr4j_no_snow.run(inputs=df)

In [7]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=gr_outputs.index, y=gr_outputs["flow"], name="GR4J + CEMANEIGE")
)
fig.add_trace(
    go.Scatter(x=gr_outputs_no_snow.index, y=gr_outputs_no_snow["flow"], name="GR4J")
)
fig.update_yaxes(title="mm/d")
fig.update_layout(
    title="Model Comparison: GR4J with vs without CEMANEIGE Preprocessing"
)
fig.show()

## CEMANEIGE with hysteresis

In [8]:
cemaneige_hysteresis_params = {
    "X1": 0.962,  # Snowpack thermal coefficient [0-1]
    "X2": 2.249,  # Melt factor [mm/(°C·day)]
    "X3": 100,
    "X4": 0.4,
}

cemaneige_hyst = CemaNeige(
    parameters=cemaneige_hysteresis_params,
    hysteresis=True,
)
snowmelt_results_hyst = cemaneige_hyst.run(df)

diff_hyst = snowmelt_results_hyst["precipitation"] - snowmelt_results["precipitation"]
print("Difference between hysteresis and non-hysteresis modes:")
print(f"  Mean: {diff_hyst.mean():.3f} mm/d")
print(f"  Max: {diff_hyst.max():.3f} mm/d")
print(f"  Min: {diff_hyst.min():.3f} mm/d")
print(f"  Total: {diff_hyst.sum():.1f} mm over period")

Difference between hysteresis and non-hysteresis modes:
  Mean: 0.000 mm/d
  Max: 14.102 mm/d
  Min: -9.029 mm/d
  Total: 0.1 mm over period


In [9]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df.index, y=snowmelt_results["precipitation"], name="Without hysteresis"
    )
)
fig.add_trace(
    go.Scatter(
        x=df.index, y=snowmelt_results_hyst["precipitation"], name="With hysteresis"
    )
)
fig.update_yaxes(title="mm/d")
fig.update_layout(title="Hysteresis Effect")
fig.show()

## CEMANEIGE multi-band

In mountainous catchments the elevation range is large enough that a single mean-basin temperature is a poor proxy for the snowpack dynamics across the watershed. CemaNeige handles this by dividing the catchment into *N* equal-area elevation bands, adjusting the input temperature per band with a lapse rate (default −0.6 °C / 100 m), running the snow model independently on each band, and returning the band-averaged liquid precipitation.

The elevation distribution of the catchment is described by a **hypsometric curve**: elevations at equal-area percentiles (0 % to 100 % of the catchment area). Here we use the La Durance at Embrun (Alps, France) catchment as an example.

In [10]:
with open(data_path / "X0310010.json") as f:
    catchment = json.load(f)

hypso = np.array(catchment["elevation_distribution"])

print(f"Catchment : {catchment['name']}")
print(f"Area      : {catchment['area']:.0f} km²")
print(
    f"Elevation : {hypso.min():.0f} – {hypso.max():.0f} m  "
    f"(range {hypso.max()-hypso.min():.0f} m, mean {hypso.mean():.0f} m)"
)

Catchment : La Durance at Embrun
Area      : 2283 km²
Elevation : 784 – 3997 m  (range 3213 m, mean 2110 m)


In [11]:
# Hypsometric curve
fig = go.Figure(
    go.Scatter(
        x=list(range(0, len(hypso))),
        y=hypso,
        mode="lines",
        fill="tozeroy",
        line=dict(color="steelblue"),
        fillcolor="rgba(70,130,180,0.2)",
    )
)
fig.update_xaxes(title="Cumulative area [%]")
fig.update_yaxes(title="Elevation [m]")
fig.update_layout(title=f"Hypsometric curve – {catchment['name']}", showlegend=False)
fig.show()

In [12]:
n_bands = 5
lapse_rate = -0.006  # °C/m

cemaneige_multi = CemaNeige(
    parameters=cemaneige_params,
    hysteresis=False,
    hypso_data=hypso,
    n_bands=n_bands,
    lapse_rate=lapse_rate,
)

mean_elev = hypso.mean()
print(f"Mean basin elevation: {mean_elev:.0f} m")
print(f"{'Band':>5}  {'Elevation':>12}  {'T offset':>10}")
print("-" * 34)
for i, (elev, offset) in enumerate(
    zip(cemaneige_multi.band_elevations, cemaneige_multi.temp_offsets)
):
    print(f"  {i+1:>3}  {elev:>9.0f} m  {offset:>+8.2f} °C")

Mean basin elevation: 2110 m
 Band     Elevation    T offset
----------------------------------
    1       1313 m     +4.79 °C
    2       1853 m     +1.55 °C
    3       2166 m     -0.34 °C
    4       2414 m     -1.82 °C
    5       2804 m     -4.16 °C


In [13]:
# Run multi-band CemaNeige
result_multi = cemaneige_multi.run(df)

# Compare with single-band result (already computed above)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=["Liquid precipitation", "Difference (multi − single)"],
)

fig.add_trace(
    go.Scatter(x=df.index, y=snowmelt_results["precipitation"], name="Single-band"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=result_multi.index, y=result_multi["precipitation"], name=f"{n_bands}-band"
    ),
    row=1,
    col=1,
)
fig.update_yaxes(title="mm/d", row=1, col=1)

diff = result_multi["precipitation"] - snowmelt_results["precipitation"]
fig.add_trace(
    go.Scatter(x=df.index, y=diff, name="Difference", line=dict(color="grey")),
    row=2,
    col=1,
)
fig.update_yaxes(title="mm/d", row=2, col=1)

fig.update_layout(
    title=f'CEMANEIGE: single-band vs {n_bands}-band ' f'({catchment["name"]})'
)
fig.show()

print(
    f"Cumulative liquid precip — single-band : {snowmelt_results['precipitation'].sum():.1f} mm"
)
print(
    f"Cumulative liquid precip — {n_bands}-band      : {result_multi['precipitation'].sum():.1f} mm"
)

Cumulative liquid precip — single-band : 37710.2 mm
Cumulative liquid precip — 5-band      : 37710.7 mm


In [14]:
# Feed multi-band output into GR4J
gr_inputs_multi = df.copy()
gr_inputs_multi["precipitation"] = result_multi["precipitation"]

gr4j_multi = ModelGr4j(parameters=gr4j_params)
gr_outputs_multi = gr4j_multi.run(inputs=gr_inputs_multi)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=gr_outputs.index, y=gr_outputs["flow"], name="Single-band CemaNeige + GR4J"
    )
)
fig.add_trace(
    go.Scatter(
        x=gr_outputs_multi.index,
        y=gr_outputs_multi["flow"],
        name=f"{n_bands}-band CemaNeige + GR4J",
    )
)
fig.update_yaxes(title="mm/d")
fig.update_layout(title=f"GR4J output: single-band vs {n_bands}-band CemaNeige")
fig.show()